In [1]:
import pandas as pd

#### Создать датафрейм из файла

In [28]:
views = pd.read_csv(
    '../data/feed-views.log',
    # skiprows=[2, 3],
    # engine='python',
    # skipfooter=2,
    names=['datetime', 'user'],
    sep='\t'
)

In [29]:
views['datetime'] = pd.to_datetime(views['datetime']).astype('datetime64[ns]')

#### Создать новые столбцы: год, месяц, день, час, минута и секунда

In [30]:
views['year'] = views['datetime'].dt.year
views['month'] = views['datetime'].dt.month
views['day'] = views['datetime'].dt.day
views['hour'] = views['datetime'].dt.hour
views['minute'] = views['datetime'].dt.minute
views['second'] = views['datetime'].dt.second

# Просмотреть результат
views.head()

,datetime,user,year,month,day,hour,minute,second
0,2020-04-17 12:01:08.463179,artem,2020,4,17,12,1,8
1,2020-04-17 12:01:23.743946,artem,2020,4,17,12,1,23
2,2020-04-17 12:27:30.646665,artem,2020,4,17,12,27,30
3,2020-04-17 12:35:44.884757,artem,2020,4,17,12,35,44
4,2020-04-17 12:35:52.735016,artem,2020,4,17,12,35,52


In [31]:
views.dtypes

datetime    datetime64[ns]
user                   str
year                 int32
month                int32
day                  int32
hour                 int32
minute               int32
second               int32
dtype: object

#### Создать столбец содержащий инфо о времени суток

In [32]:
bins = [0, 4, 7, 11, 17, 20, 24]
labels = ['night', 'early morning', 'morning', 'afternoon', 'early evening', 'evening']

views['daytime'] = pd.cut(
    views['hour'],
    bins=bins,
    labels=labels,
    right=False,
    ordered=False
)

views.set_index('user', inplace=True)

#### Рассчитать количество элементов

In [33]:
all_elements = views['year'].count()
print(f"Всего элементов в таблице:", '\n', all_elements, '\n')

by_daytime = views['daytime'].value_counts()
print("Количество элементов в категориях времени суток:")
print(by_daytime)

Всего элементов в таблице: 
 1076 

Количество элементов в категориях времени суток:
daytime
evening          509
afternoon        252
early evening    145
night            129
morning           36
early morning      5
Name: count, dtype: int64


#### Сортировка по часам, минутам и секундам в совокупности

In [34]:
views_sorted = views.sort_values(by=['hour', 'minute', 'second'])

In [35]:
views_sorted.head()

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
valentina,2020-05-15 00:00:13.222265,2020,5,15,0,0,13,night
valentina,2020-05-15 00:01:05.153738,2020,5,15,0,1,5,night
pavel,2020-05-12 00:01:27.764025,2020,5,12,0,1,27,night
pavel,2020-05-12 00:01:38.444917,2020,5,12,0,1,38,night
pavel,2020-05-12 00:01:55.395042,2020,5,12,0,1,55,night


#### Найти min, max, mode с примерами

In [36]:
views_sorted = views.sort_values(by=['hour', 'minute', 'second'])
min_hour = views['hour'].min()
max_hour = views['hour'].max()
night_max_hour = views[views['daytime'] == 'night']['hour'].max()
morning_min_hour = views[views['daytime'] == 'morning']['hour'].min()

user_night_max = views[views['hour'] == night_max_hour].index
user_morning_min = views[views['hour'] == morning_min_hour].index

print(f"Минимальный час посещения: {min_hour}", f"\nМаксимальный час посещения: {max_hour}", f"\n\nМаксимальный ночной час: {night_max_hour}", f"\nПример пользователя в это время: {user_night_max[1]}")
print(f"\nМинимальный час утром: {morning_min_hour}", f"\nПример пользователя в это время: {user_morning_min[0]}")

Минимальный час посещения: 0 
Максимальный час посещения: 23 

Максимальный ночной час: 3 
Пример пользователя в это время: konstantin

Минимальный час утром: 8 
Пример пользователя в это время: alexander


#### Найти пользователей утреннего минимального часа и ночного максимального часа

In [37]:
night_max_hour = views[views['daytime'] == 'night']['hour'].max()
morning_min_hour = views[views['daytime'] == 'morning']['hour'].min()

users_night_max = views.loc[views['hour'] == night_max_hour].index
users_morning_min = views.loc[views['hour'] == morning_min_hour].index

print("\nПользователи утренного минимального часа:", "\n".join(map(str, users_morning_min)), sep='\n')
print(f"\nПример одного пользователя в это время: {users_morning_min[0]}")

print("\nПользователи ночного максимального часа:", "\n".join(map(str, users_night_max)), sep='\n')



Пользователи утренного минимального часа:
alexander
alexander

Пример одного пользователя в это время: alexander

Пользователи ночного максимального часа:
konstantin
konstantin
konstantin


#### Найти моду для часа и для времени суток

In [38]:
hour_mode = views_sorted['hour'].mode()[0]
print(f"Мода для часа: {hour_mode}")

daytime_mode = views_sorted['daytime'].mode()[0]
print(f"Мода для времени суток: {daytime_mode}")

Мода для часа: 22
Мода для времени суток: evening


#### 3 самых ранних и самых поздних часа

In [39]:
Three_earliest = views_sorted.nsmallest(3, 'hour')[['hour']]
print(Three_earliest)

three_latest = views.nlargest(3,'hour')[['hour']]
print(three_latest)

           hour
user           
valentina     0
valentina     0
pavel         0
            hour
user            
konstantin    23
artem         23
artem         23


#### Использовать describe для получения базовой статистики по столбцам

In [ ]:

views_sorted.describe()

,datetime,year,month,day,hour,minute,second
count,1076,1076.0,1076.000000,1076.000000,1076.000000,1076.000000,1076.000000
mean,2020-05-10 09:00:41.211420672,2020.0,4.870818,13.552974,16.249071,29.629182,29.500929
min,2020-04-17 12:01:08.463179,2020.0,4.000000,1.000000,0.000000,0.000000,0.000000
25%,2020-05-10 01:13:49.857472,2020.0,5.000000,11.000000,13.000000,14.000000,14.000000
50%,2020-05-11 22:48:35.302552832,2020.0,5.000000,13.000000,19.000000,29.000000,30.000000
75%,2020-05-14 14:44:34.749530624,2020.0,5.000000,15.000000,22.000000,46.000000,45.000000
max,2020-05-22 10:36:14.662600,2020.0,5.000000,30.000000,23.000000,59.000000,59.000000
std,NaN,0.0,0.335557,4.906567,6.955490,17.689388,17.405506


In [42]:
stats = views_sorted.describe()
# print(stats)

q1 = stats.loc['25%', 'hour']
q3 = stats.loc['75%', 'hour']

iqr = q3 - q1
print(f"\nМежквартильный размах (IQR) для часов посещения: {iqr}")



Межквартильный размах (IQR) для часов посещения: 9.0
